Eтап 1. Підготовка середовища та завантаження даних

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

Етап 2: Дослідницький аналіз даних (EDA)

In [2]:
print("Перегляд перших рядків даних (df.head()):")
print(df.head())

print("Інформація про дані (df.info()):")
df.info()

print("Описова статистика (df.describe()):")
print(df.describe().T)

print("Баланс класів:")
class_counts = df['target'].value_counts()
print(class_counts)
print(f"Баланс класів: Клас 0 (Доброякісна): {class_counts[0]} | Клас 1 (Злоякісна): {class_counts[1]}")

Перегляд перших рядків даних (df.head()):
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  wor

Етап 3: Підготовка даних до моделювання

In [3]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Етап 4: Тренування та оцінка моделей

In [4]:
def evaluate_model(y_test, y_pred, model_name):
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Результати для моделі: {model_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("Classification Report:\n", classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malignant (1)']))
    return {'Model': model_name, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-score': f1}

results = [] 

print("Тренування Логістичної Регресії...")
model_lr = LogisticRegression(solver='liblinear', random_state=42)
model_lr.fit(X_train_scaled, y_train)
y_pred_lr = model_lr.predict(X_test_scaled)
results.append(evaluate_model(y_test, y_pred_lr, "Logistic Regression"))


print("Тренування SVM з лінійним ядром...")
model_svm_linear = SVC(kernel='linear', random_state=42)
model_svm_linear.fit(X_train_scaled, y_train)
y_pred_svm_linear = model_svm_linear.predict(X_test_scaled)
results.append(evaluate_model(y_test, y_pred_svm_linear, "SVM (Linear Kernel)"))

print("Тренування SVM з RBF ядром...")
model_svm_rbf = SVC(kernel='rbf', random_state=42)
model_svm_rbf.fit(X_train_scaled, y_train)
y_pred_svm_rbf = model_svm_rbf.predict(X_test_scaled)
results.append(evaluate_model(y_test, y_pred_svm_rbf, "SVM (RBF Kernel)"))

print("Тренування Випадкового Лісу (на немасштабованих даних)...")
model_rf = RandomForestClassifier(random_state=42, n_estimators=100)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
results.append(evaluate_model(y_test, y_pred_rf, "Random Forest"))

Тренування Логістичної Регресії...
Результати для моделі: Logistic Regression
Accuracy: 0.9825
Precision: 0.9861
Recall: 0.9861
F1-score: 0.9861
Classification Report:
                precision    recall  f1-score   support

   Benign (0)       0.98      0.98      0.98        42
Malignant (1)       0.99      0.99      0.99        72

     accuracy                           0.98       114
    macro avg       0.98      0.98      0.98       114
 weighted avg       0.98      0.98      0.98       114

Тренування SVM з лінійним ядром...
Результати для моделі: SVM (Linear Kernel)
Accuracy: 0.9737
Precision: 0.9859
Recall: 0.9722
F1-score: 0.9790
Classification Report:
                precision    recall  f1-score   support

   Benign (0)       0.95      0.98      0.96        42
Malignant (1)       0.99      0.97      0.98        72

     accuracy                           0.97       114
    macro avg       0.97      0.97      0.97       114
 weighted avg       0.97      0.97      0.97       1

Етап 5: Аналіз та висновки

In [5]:
results_df = pd.DataFrame(results).set_index('Model')

print("Зведена порівняльна таблиця метрик")
print(results_df.round(4))

Зведена порівняльна таблиця метрик
                     Accuracy  Precision  Recall  F1-score
Model                                                     
Logistic Regression    0.9825     0.9861  0.9861    0.9861
SVM (Linear Kernel)    0.9737     0.9859  0.9722    0.9790
SVM (RBF Kernel)       0.9825     0.9861  0.9861    0.9861
Random Forest          0.9561     0.9589  0.9722    0.9655
